In [165]:
import Data_Loader
from importlib import reload
reload(Data_Loader)
import os
import random
import torch

folder = "/mnt/c/Users/uhewm/Desktop/ProjectHGT/simulation_chunks"
#folder = "/mnt/c/ProjectHGT/simulation_chunks"

files = [entry.path for entry in os.scandir(folder) if entry.is_file()]

#if len(files) > 10000:
#    files = random.sample(files, 10000)

data_graphs = []
data = []
valid_files = []

for f in files:
    try:
        d = Data_Loader.load_file(f)
        Data_Loader.aggregate_sequences_fitch(d)
        data_graphs.append(d)
    except Exception as e:
        print(f"Fehler beim Laden von {f}: {e}")
        
    try:
        dat = Data_Loader.DonorFinder_graph_to_dataset(d, p_false = 0)
        #if len(dat) == 1:
        data.extend(dat)
        valid_files.append(f)
            
    except Exception as e:
        print(f"Fehler beim Laden von {G}: {e}")

print("Gesamtanzahl an validen Graphen: ", len(data))


eps = 1e-8
all_graph_info = torch.stack([d.graph_information for d in data])
global_min = all_graph_info.min(dim=0)[0]

shifted = torch.stack([d.graph_information - global_min for d in data])
shifted_log = torch.log1p(shifted)

global_max = shifted_log.max(dim=0)[0]

for d in data:
    gi = d.graph_information - global_min
    gi = torch.log1p(gi)
    gi = gi / (global_max + eps)
    d.graph_information = gi

"""
for node, attrs in data_graphs[0].nodes(data=True):
    seq = attrs.get("sequences", None)
    if seq is not None:
        print(node, seq[:50])   # nur die ersten 50 Stellen
"""

data_sample = random.choice(data)

/mnt/c/Users/uhewm/OneDrive/PhD/Project No.2/pangenome-gene-transfer-simulation/Data_Loader.py:96: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  edges = torch.tensor(graph_properties[1], dtype=torch.long)  # [2, num_edges]
/mnt/c/Users/uhewm/OneDrive/PhD/Project No.2/pangenome-gene-transfer-simulation/Data_Loader.py:97: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  coords = torch.tensor(graph_properties[2].T)             # [2, num_nodes]
/mnt/c/Users/uhewm/OneDrive/PhD/Project No.2/pangenome-gene-transfer-simulation/Data_Loader.py:815: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True)

Gesamtanzahl an validen Graphen:  10182


In [194]:
import torch
from torch import nn
import torch.nn.functional as F
from torch_geometric.nn import MessagePassing, global_mean_pool
import numpy as np
from collections import defaultdict
reload(Data_Loader)

class DonorFinder(nn.Module):

    def __init__(self, num_nodes, internal_node_data_dim, graph_information_dim,
                 conv_out_channels = 4, conv_kernel_size = 10, fc_hidden_channels = 512,
                 recurrent_dim = 128,
                 num_fc_layers = 4, dropout=0.2, max_number_of_snps = 300, len_alphabet = 5, recurrent_nn = True, subtree_information = False):

        super().__init__()

        #self.num_nodes = num_nodes
        self.max_number_of_snps = max_number_of_snps
        self.len_alphabet = len_alphabet
        self.internal_node_data_dim = internal_node_data_dim
        self.graph_information_dim = graph_information_dim
        self.recurrent_nn = recurrent_nn
        self.recurrent_dim = recurrent_dim
        self.subtree_information = subtree_information

        # ----- CNN Layer -----

        self.conv = nn.Conv2d(
            in_channels=1,
            out_channels=conv_out_channels,
            kernel_size=(len_alphabet, conv_kernel_size),
            stride=(1, conv_kernel_size),
            padding=0
        )

        dummy_x = torch.zeros(1, 1, len_alphabet, max_number_of_snps)
        with torch.no_grad():
            dummy_out = self.conv(dummy_x)
        dim_after_conv = dummy_out.shape
        current_dim = dim_after_conv[1] * dim_after_conv[3]
        
        # ----- LSTM Cell: Bottom-up traversal -----

        self.conv_lstm = nn.Conv2d(
            in_channels=1,
            out_channels=conv_out_channels,
            kernel_size=(len_alphabet, conv_kernel_size),
            stride=(1, conv_kernel_size),
            padding=0
        )

        current_dim = internal_node_data_dim + graph_information_dim
        self.Tree_LSTM_top_bottom = BinaryTreeLSTMCell_top_bottom(hidden_dim = recurrent_dim, 
                                                                  internal_node_data_dim = internal_node_data_dim + graph_information_dim)
        self.Tree_LSTM_bottom_top = BinaryTreeLSTMCell_bottom_top(hidden_dim = recurrent_dim, 
                                                                  internal_node_data_dim = internal_node_data_dim + graph_information_dim)
        
        if recurrent_nn:
            #current_dim = 2 * current_dim + internal_node_data_dim
            current_dim = internal_node_data_dim + 2 * recurrent_dim
        else:
            current_dim = internal_node_data_dim

        if self.subtree_information:
            current_dim = 3 * current_dim

        # ----- GCN Layer -----

        self.fusion_layer = Data_Loader.ParentChildFusionLayer(in_dim=current_dim)
        #current_dim = 6 * current_dim

        current_dim = current_dim + graph_information_dim

        # ----- Shared Fully Connected Backbone -----
        self.shared_layers = nn.ModuleList()
        self.dropout = dropout

        current_fc_hidden_channels = fc_hidden_channels
        if num_fc_layers <= 1:
            self.shared_layers.append(nn.Linear(current_dim, fc_hidden_channels))
        else:
            self.shared_layers.append(nn.Linear(current_dim, fc_hidden_channels))
            for _ in range(num_fc_layers - 1):
                self.shared_layers.append(nn.Linear(current_fc_hidden_channels, current_fc_hidden_channels // 2))
                current_fc_hidden_channels = current_fc_hidden_channels // 2
        
        # ----- Task Heads -----
        
        self.head_donor_score = nn.Linear(current_fc_hidden_channels, 1)


    def forward(self, x, internal_node_data, graph_information, level, edge_index, valid_node, batch):
        """
        Apply stacked fusion layers followed by linear classifiers.
        """      
        N = x.size(0)
        #N = self.num_nodes
        device = x.device

        num_graphs = batch.max().item() + 1
        graph_information = graph_information.view(num_graphs, -1)
        graph_information = graph_information[batch]

        #internal_node_data = (internal_node_data - mean) / std
        #internal_node_data = torch.clamp(internal_node_data, -10, 10)

        # --- Restore one-hot structure ---
        x = x.view(N, self.max_number_of_snps, self.len_alphabet)
        x = x.permute(0, 2, 1)
        x = x.unsqueeze(1)   # [N, 1, 5, max_snps]

        """
        # ----- CNN -----
        x_internal = self.conv(x)
        x_internal = F.relu(x_internal)
    
        x_internal = x_internal.squeeze(2)     # [N, C, W]
        x_internal = x_internal.flatten(start_dim=1)  # [N, C * W]
        """

        # ----- LSTM Cell: Bottom-up traversal -----

        """
        x_lstm = self.conv_lstm(x)
        x_lstm = F.relu(x_lstm)
    
        x_lstm = x_lstm.squeeze(2)     # [N, C, W]
        x_lstm = x_lstm.flatten(start_dim=1)  # [N, C * W]
        """

        x_lstm = internal_node_data
        x_lstm = torch.zeros(N, self.recurrent_dim, device=internal_node_data.device)
        #x_lstm = torch.cat((internal_node_data, graph_information), dim=1)
        
        children = self.build_children(edge_index, N)
        parent = self.build_parent(edge_index, N)

        if self.recurrent_nn:
            recurrent_data_top_down = self.tree_lstm_top_down(
                x = x_lstm,
                parent = parent,
                internal_node_data = torch.cat((internal_node_data, graph_information), dim=1),
                level = level,
                cell = self.Tree_LSTM_top_bottom
            )        
            recurrent_data_bottom_top = self.tree_lstm_bottom_up(
                x = x_lstm,
                children = children,
                internal_node_data = torch.cat((internal_node_data, graph_information), dim=1),
                level = level,
                cell = self.Tree_LSTM_bottom_top
            )
            
            #x = torch.cat((x_internal, recurrent_data, internal_node_data), dim=1)
            x = torch.cat((recurrent_data_top_down, recurrent_data_bottom_top, internal_node_data), dim=1)
        else:
            x = internal_node_data

        # ----- Subtree Layer -----

        if self.subtree_information:
            x_max_list = self.compute_subtree_representation(
                x = x,
                valid_node = valid_node,
                children = children,
                level = level,
                aggregation = "max"
            )
            x_min_list = self.compute_subtree_representation(
                x = x,
                valid_node = valid_node,
                children = children,
                level = level,
                aggregation = "min"
            )

            x_max = torch.cat([
                t if t is not None else torch.zeros(1, x.size(1), device=device)
                for t in x_max_list
            ], dim=0)
            
            x_min = torch.cat([
                t if t is not None else torch.zeros(1, x.size(1), device=device)
                for t in x_min_list
            ], dim=0)
            
            x = torch.cat((x, x_max, x_min), dim=1)
        
        # ----- GCN Layer -----

        #x = self.fusion_layer(x, edge_index)

        x = torch.cat((x, graph_information), dim=1)

        # --- GLOBAL GRAPH CONTEXT ---
        #g = global_mean_pool(x, batch)      # [num_graphs, dim]
        #x = torch.cat((x, g[batch]), dim=1) # expand global vector to nodes
                        
        # ----- Shared Backbone -----

        for layer in self.shared_layers:
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = F.relu(layer(x))

        # ----- Heads -----

        """
        real_event_logits = self.head_real_event(mean_x).view(-1)
        
        recipient_child_logits = self.head_recipient_child(mean_x).view(-1)
        
        donor_child_logits = self.head_donor_child(x).view(-1)
        
        return {
            "real_event_logits": real_event_logits,
            "recipient_child_logits": recipient_child_logits,
            "donor_child_logits": donor_child_logits
        }

        node_logits = self.head_event(x)  # [num_nodes_total, 3]
        
        return node_logits
        """
        
        node_scores = self.head_donor_score(x).view(-1)  # [num_nodes_total]

        """
        max_per_graph = torch.zeros(batch.max() + 1, device=x.device)
        max_per_graph.scatter_reduce_(
            0,
            batch,
            node_scores,
            reduce="amax",
            include_self=False
        )
        
        node_scores = node_scores - max_per_graph[batch]
        """
        
        return node_scores


    def build_children(self, edge_index, num_nodes):
        """
        edge_index: [2, E] tensor, child -> parent
        """
        children = [[] for _ in range(num_nodes)]
    
        src, dst = edge_index
        for c, p in zip(src.tolist(), dst.tolist()):
            children[p].append(c)
    
        return children

    def build_parent(self, edge_index, num_nodes):
        """
        edge_index: [2, E] child -> parent
        returns parent list of length N (root gets -1)
        """
        parent = [-1] * num_nodes
    
        src, dst = edge_index
        for c, p in zip(src.tolist(), dst.tolist()):
            parent[c] = p
    
        return parent
        
    def build_levels(self, level):
        levels = defaultdict(list)
        for v, d in enumerate(level):
            levels[int(d)].append(v)
        return levels
        

    def tree_lstm_bottom_up(self, x, children, internal_node_data, level, cell):
        """
        x: [N, D] leaf embeddings (CNN output), internal nodes arbitrary
        children: list of length N, [] or [l, r]
        level: [N] integer level values
        """
        device = x.device
        N, D = x.shape
    
        h = torch.zeros(N, D, device=device)
        c = torch.zeros(N, D, device=device)
    
        levels = self.build_levels(level)
    
        max_level = max(levels.keys())

        # level = 0 → leaves
        leaf_nodes = levels[0]
        h[leaf_nodes] = x[leaf_nodes]
    
        # process internal levels
        for d in range(1, max_level + 1):
            nodes = levels[d]
    
            # gather children indices
            left = torch.tensor(
                [children[v][0] for v in nodes],
                device=device
            )
            right = torch.tensor(
                [children[v][1] for v in nodes],
                device=device
            )
            
            node_data = internal_node_data[nodes]
    
            hl = h[left]
            cl = c[left]
            hr = h[right]
            cr = c[right]     
    
            h_new, c_new = cell(node_data, hl, cl, hr, cr)
    
            h[nodes] = h_new
            c[nodes] = c_new
   
        return h

    def tree_lstm_top_down(self, x, parent, internal_node_data, level, cell):
        """
        x: [N, D] embeddings
        parent: list length N
        level: tensor [N]
        """
    
        device = x.device
        N, D = x.shape
    
        h = torch.zeros(N, D, device=device)
        c = torch.zeros(N, D, device=device)
    
        levels = self.build_levels(level)
        max_level = max(levels.keys())
    
        # root = highest level
        root_nodes = levels[max_level]
    
        # initialize root(s)
        h[root_nodes] = x[root_nodes]
        c[root_nodes] = torch.zeros_like(h[root_nodes])
    
        # traverse downward
        for d in reversed(range(max_level)):
            nodes = levels[d]
    
            parents = torch.tensor(
                [parent[v] for v in nodes],
                device=device
            )
    
            h_parent = h[parents]
            c_parent = c[parents]
    
            node_data = internal_node_data[nodes]
    
            h_new, c_new = cell(node_data, h_parent, c_parent)
    
            h[nodes] = h_new
            c[nodes] = c_new
    
        return h

    def compute_subtree_representation(self, x, valid_node, children, level, aggregation="max"):
        """
        Computes subtree representations for every node,
        using ONLY valid nodes (valid_node == 1).
    
        x: [N, D]
        children: list of lists
        level: tensor [N]
        """
    
        device = x.device
        N, D = x.shape

        valid_mask = valid_node.view(-1).bool()
    
        levels = self.build_levels(level)
        max_level = max(levels.keys())
    
        subtree_repr = [None] * N
    
        # --- Leaves ---
        for leaf in levels[0]:
            if valid_mask[leaf]:
                subtree_repr[leaf] = x[leaf].unsqueeze(0)
            else:
                subtree_repr[leaf] = None
    
        # --- Bottom-up traversal ---
        for d in range(1, max_level + 1):
            nodes = levels[d]
    
            for v in nodes:
    
                child_list = children[v]
    
                collected = []
    
                # collect VALID child subtree tensors
                for c in child_list:
                    if subtree_repr[c] is not None:
                        collected.append(subtree_repr[c])
    
                # add current node if valid
                if valid_mask[v]:
                    collected.append(x[v].unsqueeze(0))
    
                # if no valid node in subtree
                if len(collected) == 0:
                    subtree_repr[v] = None
                    continue
    
                combined = torch.cat(collected, dim=0)
    
                if aggregation == "sum":
                    subtree_repr[v] = combined.sum(dim=0, keepdim=True)
    
                elif aggregation == "mean":
                    subtree_repr[v] = combined.mean(dim=0, keepdim=True)
    
                elif aggregation == "min":
                    subtree_repr[v] = combined.min(dim=0, keepdim=True).values
    
                elif aggregation == "max":
                    subtree_repr[v] = combined.max(dim=0, keepdim=True).values
    
                else:
                    raise ValueError("Unknown aggregation type")
    
        return subtree_repr


i = random.choice(range(100))
dummy_batch = torch.zeros(data[i].x.size(0), dtype=torch.long)

test_model = DonorFinder(
    num_nodes=data[i].x.shape[0],
    internal_node_data_dim=data[i].internal_node_data.shape[1],
    graph_information_dim=len(data[i].graph_information)
)


test_result = test_model(
    data[i].x,
    data[i].internal_node_data,
    data[i].graph_information.unsqueeze(0),
    data[i].level,
    data[i].edge_index,
    data[i].valid_node,
    dummy_batch
)

print(test_result)

tensor([0.1194, 0.1264, 0.1226, 0.1048, 0.1148, 0.1066, 0.0969, 0.1006, 0.1077,
        0.1167, 0.1237, 0.1089, 0.1189, 0.1158, 0.1047, 0.1194, 0.1011, 0.1175,
        0.1189, 0.1199, 0.1245, 0.1113, 0.1170, 0.0997, 0.1323, 0.1052, 0.1071,
        0.1245, 0.1223, 0.1161, 0.1236, 0.1234, 0.1141, 0.1302, 0.1219, 0.1185,
        0.1214, 0.1127, 0.1271, 0.1225, 0.1191, 0.1079, 0.1176, 0.1220, 0.1222,
        0.1208, 0.1178, 0.1264, 0.1280, 0.1158, 0.1189, 0.1191, 0.1180, 0.1231,
        0.1237, 0.1101, 0.1252, 0.1159, 0.1130, 0.1252, 0.1246, 0.1231, 0.1259,
        0.1229, 0.1213, 0.1194, 0.1201, 0.1198, 0.1141, 0.1293, 0.1107, 0.1250,
        0.1246, 0.1217, 0.1072, 0.1151, 0.1235, 0.1191, 0.1186, 0.1196, 0.1188,
        0.1129, 0.1145, 0.1273, 0.1177, 0.1169, 0.1150, 0.1171, 0.1185, 0.1236,
        0.1312, 0.1196, 0.1169, 0.1090, 0.1115, 0.1224, 0.1075, 0.1196, 0.1154,
        0.1191, 0.1224, 0.0963, 0.1214, 0.1199, 0.1108, 0.1226, 0.1194, 0.1205,
        0.1069, 0.1265, 0.1190, 0.1194, 

In [195]:
import Data_Loader
reload(Data_Loader)
from torch_geometric.loader import DataLoader
from collections import Counter
import NN_visualization
reload(NN_visualization)

folder_save = "/mnt/c/Users/uhewm/Desktop/ProjectHGT"
Path(folder_save).mkdir(parents=True, exist_ok=True)  # sicherstellen, dass der Ordner existiert

# === 1. Modell, Optimizer, Loss ===

model = DonorFinder(
    num_nodes = data[0].x.shape[0], 
    internal_node_data_dim = data[0].internal_node_data.shape[1],
    graph_information_dim = len(data[0].graph_information)
)

optimizer = torch.optim.Adam(model.parameters(), lr = 5*1e-4)

# === 2. Training & Evaluation ===

# Train/Test Split
random.shuffle(data)
split_idx = int(0.3 * len(data))
train_data = data[:split_idx]
test_data = data[split_idx:]

train_loader = DataLoader(train_data, batch_size=8, shuffle=True)
test_loader = DataLoader(test_data, batch_size=8)

# Training weights:
counter = Counter()
for d in train_data:
    counter.update(d.event_label.tolist())

def train():
    model.train()
    total_loss = 0.0

    for batch in train_loader:
        optimizer.zero_grad()

        node_scores = model(
            batch.x,
            batch.internal_node_data,
            batch.graph_information,
            batch.level,
            batch.edge_index,
            batch.valid_node,
            batch.batch
        )  # [num_nodes_total]

        if torch.isnan(node_scores).any():
            print("NaN in model output!")
            exit()
    
        loss = donor_selection_loss(node_scores, batch)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(train_loader)

def donor_selection_loss(node_scores, batch):

    device = node_scores.device
    loss = torch.zeros(1, device=device)

    for graph_id in batch.batch.unique():

        mask = (batch.batch == graph_id)

        scores = node_scores[mask]
        labels = batch.event_label[mask]
        donor_mask = batch.possible_donor_mask[mask].squeeze()

        scores = scores.clone()
        scores[donor_mask == 0] = -1e9

        donor_indices = (labels > 0).nonzero(as_tuple=True)[0]

        # --- CASE 1: kein echtes Event ---
        if len(donor_indices) == 0:

            # hier sollten alle Scores niedrig sein
            targets = torch.zeros_like(scores)
            loss += F.binary_cross_entropy_with_logits(scores, targets)
            continue

        # --- CASE 2: echtes Event ---
        true_idx = donor_indices[0]

        log_probs = scores - torch.logsumexp(scores, dim=0)

        ce_loss = -log_probs[true_idx]

        loss += ce_loss

    return loss / batch.num_graphs
    
@torch.no_grad()
def evaluate_loss(loader):
    model.eval()
    total_loss = 0.0

    for batch in loader:
        node_scores = model(
            batch.x,
            batch.internal_node_data,
            batch.graph_information,
            batch.level,
            batch.edge_index,
            batch.valid_node,
            batch.batch
        )

        loss = donor_selection_loss(node_scores, batch)
        total_loss += loss.item()

    return total_loss / len(loader)
    
@torch.no_grad()
def evaluate_accuracy(loader):

    model.eval()

    total_graphs = 0
    correct = 0

    for batch in loader:

        node_scores = model(
            batch.x,
            batch.internal_node_data,
            batch.graph_information,
            batch.level,
            batch.edge_index,
            batch.valid_node,
            batch.batch
        )

        for graph_id in batch.batch.unique():

            total_graphs += 1
            mask = (batch.batch == graph_id)

            scores = node_scores[mask].clone()
            labels = batch.event_label[mask]
            donor_mask = batch.possible_donor_mask[mask].squeeze()

            scores[donor_mask == 0] = -1e9

            donor_indices = (labels > 0).nonzero(as_tuple=True)[0]

            if len(donor_indices) == 0:
                continue

            true_idx = donor_indices[0]
            pred_idx = scores.argmax()

            if pred_idx == true_idx:
                correct += 1

    return correct / total_graphs    

patience = 10
min_delta = 1e-5
max_epochs = 500

best_val_loss = float("inf")
epochs_no_improve = 0
best_model_state = None

for epoch in range(1, max_epochs + 1):

    train_loss = train()
    val_loss = evaluate_loss(test_loader)
    train_acc = evaluate_accuracy(train_loader)
    val_acc = evaluate_accuracy(test_loader)

    print(
        f"Epoch {epoch:03d} | "
        f"Train: {train_loss:.4f} | "
        f"TrainAcc: {train_acc:.4f} | "
        f"Val: {val_loss:.4f} | "
        f"ValAcc: {val_acc:.4f}"
    )

    # --- Check improvement ---
    if val_loss < best_val_loss - min_delta:
        best_val_loss = val_loss
        epochs_no_improve = 0
        best_model_state = model.state_dict()
    else:
        epochs_no_improve += 1

    # --- Early stopping ---
    if epochs_no_improve >= patience and epoch >= 40:
        print(f"\nEarly stopping triggered at epoch {epoch}")
        break

    NN_visualization.save_model(model, print_save_path = False)

# Restore best model
if best_model_state is not None:
    model.load_state_dict(best_model_state)
    NN_visualization.save_model(model, print_save_path = False)
    print("Best model restored.")

Epoch 001 | Train: 3.0975 | TrainAcc: 0.1637 | Val: 2.8004 | ValAcc: 0.1714
Model saved to /mnt/c/Users/uhewm/Desktop/ProjectHGT/donorfinder_model.pt
Epoch 002 | Train: 2.8506 | TrainAcc: 0.1781 | Val: 2.7171 | ValAcc: 0.1835
Model saved to /mnt/c/Users/uhewm/Desktop/ProjectHGT/donorfinder_model.pt
Epoch 003 | Train: 2.7403 | TrainAcc: 0.1916 | Val: 2.6950 | ValAcc: 0.1921
Model saved to /mnt/c/Users/uhewm/Desktop/ProjectHGT/donorfinder_model.pt
Epoch 004 | Train: 2.6468 | TrainAcc: 0.2112 | Val: 2.5628 | ValAcc: 0.2033
Model saved to /mnt/c/Users/uhewm/Desktop/ProjectHGT/donorfinder_model.pt
Epoch 005 | Train: 2.5634 | TrainAcc: 0.2200 | Val: 2.4751 | ValAcc: 0.2127
Model saved to /mnt/c/Users/uhewm/Desktop/ProjectHGT/donorfinder_model.pt
Epoch 006 | Train: 2.5432 | TrainAcc: 0.2354 | Val: 2.4408 | ValAcc: 0.2327
Model saved to /mnt/c/Users/uhewm/Desktop/ProjectHGT/donorfinder_model.pt
Epoch 007 | Train: 2.4765 | TrainAcc: 0.2377 | Val: 2.4416 | ValAcc: 0.2308
Model saved to /mnt/c/Us

In [199]:
global_max

tensor([1.9196, 0.6533, 1.1352, 0.3097, 1.1352, 2.1918, 2.9805, 5.5094, 5.5294,
        5.5373, 6.2577, 5.5094, 5.4806, 0.0000, 0.0000, 5.4765, 0.0000, 0.0000,
        5.5452, 6.2500, 5.5094, 5.5568, 6.2729, 5.5373, 5.0752, 6.1841, 5.4806,
        5.0752, 6.1759, 5.4765, 5.6630, 6.2841, 5.5452])

In [200]:
import NN_visualization
reload(NN_visualization)

folder_save = "/mnt/c/Users/uhewm/Desktop/ProjectHGT"
Path(folder_save).mkdir(parents=True, exist_ok=True)  # sicherstellen, dass der Ordner existiert

NN_visualization.save_DonorFinder(model, global_max = global_max, global_min = global_min, eps = eps)

model_save = model

Model saved to /mnt/c/Users/uhewm/Desktop/ProjectHGT/donorfinder_model.pt


In [153]:
def x_in_train(single_data, train_data):

    for d in train_data:
        if torch.equal(single_data.time_dist_from_recipient, d.time_dist_from_recipient):
            return True

    return False

while True:
    random_file = random.choice(valid_files)
    #random_file = file 
    d = Data_Loader.load_file(random_file)
    single_graph = Data_Loader.aggregate_sequences_fitch(d)
    single_data = Data_Loader.DonorFinder_graph_to_dataset(single_graph, p_false = 0)
    
    if len(single_data) == 1: # > 0:
        single_data = single_data[random.choice(range(len(single_data)))]
        #single_data = single_data[0]
        if sum(single_data.event_label) > 0: # real event
            print(random_file)
            break

x_in_train(single_data, train_data)

/mnt/c/Users/uhewm/Desktop/ProjectHGT/simulation_chunks/simulation_5906.h5


True

In [164]:
import torch
from pyvis.network import Network
import subprocess
import webbrowser
from pathlib import Path
import networkx as nx

reload(Data_Loader)

while True:
    random_file = random.choice(valid_files)
    #random_file = file 
    d = Data_Loader.load_file(random_file)
    single_graph = Data_Loader.aggregate_sequences_fitch(d)
    single_data = Data_Loader.DonorFinder_graph_to_dataset(single_graph, p_false = 0)
    
    if len(single_data) > 0: #== 1: # > 0:
        single_data = single_data[random.choice(range(len(single_data)))]
        #single_data = single_data[0]
        if sum(single_data.event_label) > 0 and not x_in_train(single_data, train_data): # real event
            break
            
gi = single_data.graph_information - global_min
gi = torch.log1p(gi)
gi = gi / (global_max + eps)
single_data.graph_information = gi

recipient = (single_data.time_dist_from_recipient[:, 0] + single_data.time_dist_from_recipient[:, 1] == 0) .nonzero(as_tuple=True)[0].item()
donor = (single_data.time_dist_from_donor[:, 0] + single_data.time_dist_from_donor[:, 1] == 0) .nonzero(as_tuple=True)[0].item()
donor_parent = list(single_graph.predecessors(donor))[0]
recipient_child = (
    sorted(list(single_graph.successors(recipient)))[0] if sum(single_data.event_label) == 1
    else sorted(list(single_graph.successors(recipient)))[1] if sum(single_data.event_label) == 2
    else None
)
print(donor, donor_parent)

hgt_nodes = [node for node in single_graph.nodes() if single_graph.nodes[node]["recipient"]["is_parent_node"] ]
gene_absence_presence_matrix = [single_graph.nodes[node]["gene_present_below_node"] > 0 for node in single_graph.nodes() if node < 100]
child_sim = single_data.child_similarity.cpu().numpy()

# === 1. Modellvorhersagen berechnen ===
model.eval()
with torch.no_grad():
    node_scores = model(
        single_data.x,
        single_data.internal_node_data,
        single_data.graph_information.unsqueeze(0),
        single_data.level,
        single_data.edge_index,
        single_data.valid_node,
        torch.zeros(single_data.x.size(0), dtype=torch.long)  # single graph batch
    )

    # --- Wichtige Maskierung: nur mögliche Donoren ---
    donor_mask = single_data.possible_donor_mask.squeeze()
    scores_masked = node_scores.clone()
    scores_masked[donor_mask == 0] = -1e9

    # Softmax über gültige Donoren
    probs = torch.softmax(scores_masked, dim=0).cpu().numpy()

# Map von Node-ID zu Wahrscheinlichkeit
pred_idx = int(torch.argmax(node_scores))
predicted_donor_node = list(single_graph.nodes())[pred_idx]

pred_probs = {node: float(probs[i]) for i, node in enumerate(single_graph.nodes())}
child_sim_map = {node: child_sim[i] for i, node in enumerate(single_graph.nodes())}

true_recipient_parent_nodes = [
    node for node in single_graph.nodes()
    if single_graph.nodes[node]["recipient"].get("is_parent_node", False)
]

# --- Top-5 Donor Nodes bestimmen ---
sorted_probs = sorted(pred_probs.items(), key=lambda x: x[1], reverse=True)
top5_donor_nodes = [node for node, prob in sorted_probs[:5] if prob > 0]

# --- x/y Koordinaten für Blätter und innere Knoten berechnen ---
x_spacing = 100
y_spacing = 100

node_x = {}
node_y = {}

# Maximaler Level aus single_graph
max_level = max(single_graph.nodes[n].get("level", 0) for n in single_graph.nodes)

# Hilfsfunktion: finde alle Blätter unterhalb eines Knotens
def get_descendant_leaves(G, node):
    """Alle Blätter, die von `node` erreichbar sind (rekursiv)."""
    stack = list(G.successors(node))
    reachable_leaves = []
    while stack:
        temp_node = stack.pop()
        children = list(G.successors(temp_node))
        if len(children) > 0:
            stack.extend(children)
        else:
            reachable_leaves.append(temp_node)
    return reachable_leaves

# === Blätter (Level 0) oben ===
leaves = [n for n in single_graph.nodes if single_graph.nodes[n].get("level", 0) == 0]
for i, node in enumerate(sorted(leaves)):  
    node_x[node] = i * x_spacing
    node_y[node] = (max_level - 0) * y_spacing  # Blätter oben

# === Innere Knoten: levelweise platzieren ===
levels_in_graph = sorted(set(nx.get_node_attributes(single_graph, "level").values()))
for level in levels_in_graph[1:]:  # 0 schon behandelt
    nodes_in_level = [n for n in single_graph.nodes if single_graph.nodes[n].get("level", 0) == level]
    for node in nodes_in_level:
        # Finde alle Blätter unterhalb
        reachable_leaves = get_descendant_leaves(single_graph, node)
        if reachable_leaves:
            leaf_x = [node_x[l] for l in reachable_leaves if l in node_x]
            node_x[node] = np.mean(leaf_x)
        else:
            node_x[node] = 0
        node_y[node] = (max_level - level) * y_spacing
   
# === Netzwerk initialisieren (Hierarchical Layout deaktiviert!) ===
net = Network(height="900px", width="100%", directed=True)

net.set_options("""
{
  "nodes": {
    "shape": "dot",
    "size": 12,
    "font": { "size": 24 }
  },
  "edges": {
    "arrows": {
      "to": { "enabled": true, "scaleFactor": 0.5 }
    }
  },
  "physics": {
    "enabled": false
  }
}
""")
# === Knoten hinzufügen mit festen x/y ===

ATTRS = [
    "sum_seq",
    "tree_length",
    "time",
    "time_only_valid_nodes",
    "time_dist_from_recipient",
    "valid_node"
]

for node in single_graph.nodes():
    values = {}

    for key in ATTRS:
        if key == "pred":
            values[key] = float(pred_probs[node])
        else:
            values[key] = float(single_graph.nodes[node].get(key, 0))

    # Title-String
    title = ", ".join([f"{k}: {values[k]:.2f}" for k in ATTRS])

    pred_value = pred_probs[node]

    #label = f"{node}\nPred: {pred_value:.3f}"
    level = single_graph.nodes[node].get("level", 0)
    if level > 0:
        sim_vals = child_sim_map[node]
    
        label = (
            f"{node}\n"
            f"Pred: {pred_value:.3f}"
            f"  Sim/Dif: {int(sim_vals[0])} {int(sim_vals[1])} {int(sim_vals[2])} {int(sim_vals[3])} {int(sim_vals[4])} {int(sim_vals[5])}"
        )
    else:
        label = f"{node}"

    # Farbe
    #if node == donor:
    #    color = "darkred"                        # echter Donor      
    if node in top5_donor_nodes:
        color = "red"  # Top-5 vorhergesagte Donoren
    elif node in true_recipient_parent_nodes:
        color = "blue"                       # other Recipient Parent  
    #elif node == predicted_donor_node:
    #    color = "cyan"                       # vorhergesagter Donor
    elif node < 100 and gene_absence_presence_matrix[node] == 1:
        color = "orange"
    elif node < 100 and gene_absence_presence_matrix[node] == 0:
        color = "black"
    else:
        color = "lightblue"
    
    #color = blend_with_color(color, pred_value)

    net.add_node(node, label=label, title=title, color=color,
                 x=node_x[node], y=node_y[node])


# === Kanten hinzufügen ===
for u, v in single_graph.edges():
    net.add_edge(u, v)

# Add an edge from donor to recipient (i.e. the hgt edge)
mid_donor_x = (node_x[donor] + node_x[donor_parent]) / 2
mid_donor_y = (node_y[donor] + node_y[donor_parent]) / 2
mid_donor_id = "Donor"
mid_recipient_x = (node_x[recipient] + node_x[recipient_child]) / 2
mid_recipient_y = (node_y[recipient] + node_y[recipient_child]) / 2
mid_recipient_id = "Recipient"
net.add_node(mid_donor_id, label="", color="rgba(0,0,0,0)", size=1, x=mid_donor_x, y=mid_donor_y, physics=False)
net.add_node(mid_recipient_id, label="", color="rgba(0,0,0,0)", size=1, x=mid_recipient_x, y=mid_recipient_y, physics=False)
net.add_edge(mid_donor_id, mid_recipient_id, color="red", arrows="to", width=5)

# === HTML-Datei speichern und direkt in Chrome öffnen ===
html_file = Path("/mnt/c/Users/uhewm/OneDrive/PhD/Project No.2/pangenome/graph.html")
html_file = Path("/mnt/c/ProjectHGT/graph.html")
html_file.parent.mkdir(parents=True, exist_ok=True)
net.show(str(html_file), notebook=False)

# WSL-Pfad in Windows-Pfad umwandeln
win_path = subprocess.run(["wslpath", "-w", str(html_file)], capture_output=True, text=True).stdout.strip()

# Direkt in Chrome öffnen
subprocess.run(["cmd.exe", "/C", "start", "chrome", win_path])


/mnt/c/Users/uhewm/OneDrive/PhD/Project No.2/pangenome-gene-transfer-simulation/Data_Loader.py:96: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  edges = torch.tensor(graph_properties[1], dtype=torch.long)  # [2, num_edges]
/mnt/c/Users/uhewm/OneDrive/PhD/Project No.2/pangenome-gene-transfer-simulation/Data_Loader.py:97: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  coords = torch.tensor(graph_properties[2].T)             # [2, num_nodes]
/mnt/c/Users/uhewm/OneDrive/PhD/Project No.2/pangenome-gene-transfer-simulation/Data_Loader.py:795: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True)

197 198
/mnt/c/ProjectHGT/graph.html
>4;1H84l=                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       

CompletedProcess(args=['cmd.exe', '/C', 'start', 'chrome', 'C:\\ProjectHGT\\graph.html'], returncode=0)

In [172]:
data[0].internal_node_data[142]

tensor([0.0000, 0.0234, 0.2801, 0.2930, 1.0000, 0.9963, 0.9985, 1.0000, 0.8451,
        0.9472, 0.9765, 0.9734, 0.9765, 0.9794, 0.9786, 0.9794, 0.0133, 0.0133,
        0.0000, 0.0000, 1.0000, 1.0000])

In [4]:
import matplotlib.colors as mcolors

def blend_with_color(base_color, pred, threshold=0.1):
    """
    pred nahe 0  -> ursprüngliche Farbe
    pred nahe 1  -> stark rot
    """
    if pred <= threshold:
        return base_color

    strength = (pred - threshold) / (1 - threshold)
    strength = max(0.0, min(1.0, strength))

    base_rgb = mcolors.to_rgb(base_color)
    red_rgb = (1, 0, 0)

    blended = tuple(
        (1 - strength) * base_rgb[i] + strength * red_rgb[i]
        for i in range(3)
    )

    return mcolors.to_hex(blended)

In [193]:
import torch
from torch import nn

class BinaryTreeLSTMCell_bottom_top(nn.Module):
    """
    Binary Tree-LSTM cell (Tai et al., 2015) from leafs to root.
    """

    def __init__(self, hidden_dim, internal_node_data_dim):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.internal_node_data_dim = internal_node_data_dim
        
        def lin():
            return nn.Linear(hidden_dim + internal_node_data_dim, hidden_dim)

        self.i = lin()
        self.f = lin()
        self.u = lin()
        self.o = lin()

    def forward(self, x, hl, cl, hr, cr):
        """
        hl, cl: left child hidden & cell
        hr, cr: right child hidden & cell

        The left and right order does not matter (invariant).
        """

        hsum = hl + hr

        i = torch.sigmoid(self.i(torch.cat((x, hsum), dim=1)))
        fl = torch.sigmoid(self.f(torch.cat((x, hl), dim=1)))
        fr = torch.sigmoid(self.f(torch.cat((x, hr), dim=1)))
        o = torch.sigmoid(self.o(torch.cat((x, hsum), dim=1)))
        u = torch.tanh(self.u(torch.cat((x, hsum), dim=1)))

        c = i * u + fl * cl + fr * cr

        h = o * torch.tanh(c)

        return h, c

class BinaryTreeLSTMCell_top_bottom(nn.Module):
    """
    Binary Tree-LSTM cell (Tai et al., 2015) from root to leafs.
    """

    def __init__(self, hidden_dim, internal_node_data_dim):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.internal_node_data_dim = internal_node_data_dim
        
        def lin():
            return nn.Linear(hidden_dim + internal_node_data_dim, hidden_dim)

        self.i = lin()
        self.f = lin()
        self.u = lin()
        self.o = lin()

    def forward(self, x, h_parent, c_parent):

        i = torch.sigmoid(self.i(torch.cat((x, h_parent), dim=1)))
        f = torch.sigmoid(self.f(torch.cat((x, h_parent), dim=1)))
        o = torch.sigmoid(self.o(torch.cat((x, h_parent), dim=1)))
        u = torch.tanh(self.u(torch.cat((x, h_parent), dim=1)))

        c = i * u + f * c_parent
        h = o * torch.tanh(c)

        return h, c
